# Consensus Strategy Evaluation

**Strategy:** DNN picks direction + edge >= 0.07, confirmed by majority (2/3) of LR/RF/XGB.

**Method:** Load 4 exported production models, generate predictions on unseen 20%, simulate betting with 7.2% fees.

**Output:** `data/optimal_strategy_consensus.json`

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

TAKER_FEE_RATE = 0.072
MAX_BID = 0.85
PRED_DIR = Path("../../data/consensus_preds")
PRED_DIR.mkdir(exist_ok=True)

## 1. Generate Predictions via Subprocess

In [ ]:
PREDICT_SCRIPT = """
import gc, json, sys, os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

model_name = sys.argv[1]
pred_dir = sys.argv[2]

print(f"  Loading data...", flush=True)
df = pd.read_json("../../data/latest_features.jsonl", lines=True)
df["target"] = (df["outcome"] == "UP").astype(int)
cids = df["candle_id"].unique()
split = int(len(cids) * 0.8)
val_ids = set(cids[split:])
df_val = df[df["candle_id"].isin(val_ids)]
del df; gc.collect()

BASE_COLS = [
    "candle_id", "timestamp", "outcome", "target", "elapsed_pct",
    "up_best_bid", "up_best_ask", "up_bid_depth", "up_ask_depth",
    "down_best_bid", "down_best_ask", "down_bid_depth", "down_ask_depth",
]

# Save base columns once
base_path = os.path.join(pred_dir, "val_base.csv")
if not os.path.exists(base_path):
    df_val[BASE_COLS].to_csv(base_path, index=True)
    print(f"  Base columns saved", flush=True)

if model_name == "lr":
    feat_cols = joblib.load("../../models/feature_cols_v1.joblib")
    scaler = joblib.load("../../models/scaler_v1.joblib")
    model = joblib.load("../../models/logistic_v1.joblib")
    X = scaler.transform(df_val[feat_cols].fillna(0).values)
    probs = model.predict_proba(X)[:, 1]

elif model_name == "rf":
    feat_cols = joblib.load("../../models/rf_feature_cols_v1.joblib")
    scaler = joblib.load("../../models/rf_scaler_v1.joblib")
    model = joblib.load("../../models/rf_v1.joblib")
    X = scaler.transform(df_val[feat_cols].fillna(0).values)
    probs = model.predict_proba(X)[:, 1]

elif model_name == "xgb":
    feat_cols = joblib.load("../../models/xgb_feature_cols_v1.joblib")
    scaler = joblib.load("../../models/xgb_scaler_v1.joblib")
    model = joblib.load("../../models/xgb_calibrator_v1.joblib")
    X = scaler.transform(df_val[feat_cols].fillna(0).values)
    probs = model.predict_proba(X)[:, 1]

elif model_name == "dnn":
    import torch
    feat_cols = joblib.load("../../models/dnn_feature_cols_v1.joblib")
    scaler = joblib.load("../../models/dnn_scaler_v1.joblib")
    model = torch.load("../../models/dnn_v1.pt", weights_only=False)
    model.eval()
    cal_path = "../../models/dnn_calibrator_v1.joblib"
    calibrator = joblib.load(cal_path) if os.path.exists(cal_path) else None
    X = scaler.transform(df_val[feat_cols].fillna(0).values)
    with torch.no_grad():
        raw = torch.sigmoid(model(torch.tensor(X.astype(np.float32)))).numpy().flatten()
    probs = calibrator.predict(raw) if calibrator else raw

pd.DataFrame({f"prob_{model_name}": probs}, index=df_val.index).to_csv(
    os.path.join(pred_dir, f"val_{model_name}.csv"), index=True
)
print(f"  {model_name}: {len(probs):,} predictions saved", flush=True)
"""

# Write helper script
script_path = PRED_DIR / "_predict.py"
script_path.write_text(PREDICT_SCRIPT)

# Run each model in subprocess
for name in ["lr", "rf", "xgb", "dnn"]:
    print(f"=== {name.upper()} ===", flush=True)
    result = subprocess.run(
        [sys.executable, str(script_path), name, str(PRED_DIR)],
        cwd=str(Path(".").resolve()),
    )
    if result.returncode != 0:
        raise RuntimeError(f"{name} prediction failed")
    print(flush=True)

## 2. Load Predictions & Simulate

In [ ]:
# Load
base = pd.read_csv(PRED_DIR / "val_base.csv", index_col=0)
for m in ["lr", "rf", "xgb", "dnn"]:
    preds = pd.read_csv(PRED_DIR / f"val_{m}.csv", index_col=0)
    base = base.join(preds)
df_val = base
print(f"Val: {len(df_val):,} snapshots, {df_val['candle_id'].nunique():,} candles")

# Precompute
dir_lr = (df_val["prob_lr"].values >= 0.5).astype(int)
dir_rf = (df_val["prob_rf"].values >= 0.5).astype(int)
dir_xgb = (df_val["prob_xgb"].values >= 0.5).astype(int)
dir_dnn = (df_val["prob_dnn"].values >= 0.5).astype(int)
conf_dnn = np.maximum(df_val["prob_dnn"].values, 1 - df_val["prob_dnn"].values)
ask_dnn = np.where(dir_dnn == 1, df_val["up_best_ask"].values, df_val["down_best_ask"].values)
edge_dnn = conf_dnn - ask_dnn
maj_dir_3 = np.where((dir_lr + dir_rf + dir_xgb) >= 2, 1, 0)


def simulate(df_preds, decisions, name):
    BET = 10.0
    balance = 1000.0
    wins = total = 0
    peak = balance
    max_dd = 0.0
    pnls = []
    for cid in df_preds["candle_id"].unique():
        mask = df_preds["candle_id"] == cid
        idxs = df_preds.index[mask]
        outcome = df_preds.loc[idxs[0], "outcome"]
        cpnl = 0.0
        entered = False
        for idx in idxs:
            d = decisions[df_preds.index.get_loc(idx)]
            if d == 0 or entered or balance < BET:
                continue
            direction = "UP" if d == 1 else "DOWN"
            ask = df_preds.loc[idx, "up_best_ask"] if d == 1 else df_preds.loc[idx, "down_best_ask"]
            if ask is None or not np.isfinite(ask) or ask <= 0 or ask >= MAX_BID:
                continue
            entered = True
            total += 1
            won = direction == outcome
            if won:
                gross = BET / ask
                fee = gross * TAKER_FEE_RATE * ask * (1 - ask)
                profit = (gross - fee) - BET
                balance += profit
                cpnl += profit
                wins += 1
            else:
                balance -= BET
                cpnl -= BET
            if balance > peak:
                peak = balance
            dd = (peak - balance) / peak if peak > 0 else 0
            if dd > max_dd:
                max_dd = dd
        pnls.append(cpnl)
    wr = wins / total * 100 if total else 0
    ret = (balance - 1000) / 1000 * 100
    pnl_arr = np.array(pnls)
    sharpe = float(pnl_arr.mean() / pnl_arr.std()) if len(pnl_arr) >= 2 and pnl_arr.std() > 0 else 0
    print(
        f"{name:<40} Bets={total:>5}  WR={wr:>5.1f}%  Return={ret:>+7.1f}%  MaxDD={max_dd * 100:>5.1f}%  Sharpe={sharpe:>+.4f}  Final=${balance:>.0f}"
    )
    return {"bets": total, "wr": wr, "return": ret, "max_dd": max_dd * 100, "sharpe": sharpe}


def make_decisions(edge_thresh, require_agree=None):
    d = np.zeros(len(df_val), dtype=int)
    dnn_bet = edge_dnn >= edge_thresh
    if require_agree == "majority":
        mask = dnn_bet & (maj_dir_3 == dir_dnn)
    else:
        mask = dnn_bet
    d[mask & (dir_dnn == 1)] = 1
    d[mask & (dir_dnn == 0)] = 2
    return d


def static_decisions(prob_col, edge_threshold=0.05):
    p = df_val[prob_col].values
    conf = np.maximum(p, 1 - p)
    d_dir = (p >= 0.5).astype(int)
    ask = np.where(d_dir == 1, df_val["up_best_ask"].values, df_val["down_best_ask"].values)
    edge = conf - ask
    decisions = np.zeros(len(df_val), dtype=int)
    bet = edge >= edge_threshold
    decisions[bet & (d_dir == 1)] = 1
    decisions[bet & (d_dir == 0)] = 2
    return decisions


print("=" * 110)
print(f"{'Strategy':<40} {'Bets':>5}  {'WR':>6}  {'Return':>8}  {'MaxDD':>6}  {'Sharpe':>7}  {'Final':>6}")
print("=" * 110)
r_consensus = simulate(df_val, make_decisions(0.07, "majority"), "Consensus (DNN+maj, edge>=0.07)")
simulate(df_val, make_decisions(0.05), "DNN alone (edge>=0.05)")
print("-" * 110)
for col, nm in [("prob_lr", "LR"), ("prob_rf", "RF"), ("prob_xgb", "XGB"), ("prob_dnn", "DNN")]:
    simulate(df_val, static_decisions(col, 0.05), f"Static edge>=0.05 ({nm})")
print("=" * 110)

## 3. Export Strategy Config

In [ ]:
from datetime import UTC, datetime

config = {
    "model": "consensus",
    "strategy": "DNN + majority agree",
    "min_edge": 0.05,
    "max_entries": 1,
    "min_btc_move": 0.0,
    "edge_threshold": 0.07,
    "min_agreement": 2,
    "dnn_name": "DNN",
    "sharpe": round(r_consensus["sharpe"], 4),
    "win_rate": round(r_consensus["wr"] / 100, 4),
    "return_pct": round(r_consensus["return"], 2),
    "max_drawdown": round(r_consensus["max_dd"] / 100, 4),
    "total_bets": r_consensus["bets"],
    "eval_method": "production_models_80_20_split",
    "created_at": datetime.now(UTC).isoformat(),
}

out_path = Path("../../data/optimal_strategy_consensus.json")
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved to {out_path}")
print(json.dumps(config, indent=2))

## 4. Equity Curve

In [ ]:
import matplotlib.pyplot as plt

curves = {
    "Consensus (DNN+maj, 0.07)": make_decisions(0.07, "majority"),
    "DNN alone (0.05)": make_decisions(0.05),
}

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#4488ff", "#ff4444"]

for (name, decisions), color in zip(curves.items(), colors, strict=True):
    BET = 10.0
    balance = 1000.0
    history = [balance]
    for cid in df_val["candle_id"].unique():
        mask = df_val["candle_id"] == cid
        idxs = df_val.index[mask]
        outcome = df_val.loc[idxs[0], "outcome"]
        entered = False
        for idx in idxs:
            d = decisions[df_val.index.get_loc(idx)]
            if d == 0 or entered or balance < BET:
                continue
            direction = "UP" if d == 1 else "DOWN"
            ask = df_val.loc[idx, "up_best_ask"] if d == 1 else df_val.loc[idx, "down_best_ask"]
            if ask is None or not np.isfinite(ask) or ask <= 0 or ask >= MAX_BID:
                continue
            entered = True
            if direction == outcome:
                gross = BET / ask
                fee = gross * TAKER_FEE_RATE * ask * (1 - ask)
                balance += (gross - fee) - BET
            else:
                balance -= BET
        history.append(balance)
    ax.plot(history, label=f"{name}  ->  ${history[-1]:,.0f}", color=color, linewidth=1.5)
    ax.annotate(
        f"${history[-1]:,.0f}",
        xy=(len(history) - 1, history[-1]),
        xytext=(8, 0),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        color=color,
        va="center",
    )

ax.axhline(y=1000, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Candles")
ax.set_ylabel("Balance ($)")
ax.set_title("Consensus vs DNN Alone (7.2% fee, $10 flat)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.2)
ax.margins(x=0.08)
fig.tight_layout()
plt.show()